# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the:
**Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset, using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`. This step will download and parse the dataset description and schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

print(f"Dataset loaded: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
We examine the available record sets, fields, and their `@id`. These identifiers are essential for programmatic access and future referencing.

In [ ]:
# List all available record sets (@id) in the dataset
print("Available record sets and their field IDs:")
all_record_sets = dataset.metadata.record_sets
for record_set in all_record_sets:
    print(f"- Record set @id: {record_set.id}  Name: {getattr(record_set, 'name', None)}")
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            print(f"    - Field @id: {field.id}  Name: {getattr(field, 'name', None)}  DataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Here, we load data from one or more record sets into DataFrames for analysis. Use record set and field `@id`s as shown in the Overview.

In [ ]:
# Gather all available record set IDs
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set into a pandas DataFrame
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")
    else:
        print(f"No records found for record set {record_set_id}")

# Display columns of the first record set with data
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"Record set {first_rs} columns: {dataframes[first_rs].columns.tolist()}")
    dataframes[first_rs].head()
else:
    print('No record sets with data found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and basic grouping. We demonstrate this on the first loaded record set, and always reference fields by their `@id`.

In [ ]:
import numpy as np

# Select the first available record set with data
record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        record_set_id = rs_id
        break

if record_set_id is not None:
    df = dataframes[record_set_id]

    # List numeric fields (by checking dtype)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields for record set '{record_set_id}': {numeric_fields}")

    # Try to select the first numeric field available, or set manually here using `@id`
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        
        # Filter: For demonstration, keep only values above median
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records in {record_set_id} with {numeric_field} > {threshold:.3f}:")
        print(filtered_df[[numeric_field]].head())

        # Normalize
        filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

        # Try groupby on a categorical field if available
        categorical_fields = [col for col in df.columns if df[col].dtype == object]
        if categorical_fields:
            group_field = categorical_fields[0]
            print(f"Grouping by {group_field}:\n")
            # Safeguard against non-numeric group means
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped.head())
        else:
            print("No categorical fields available for groupby.")
    else:
        print(f"No numeric fields in first record set {record_set_id}.")
else:
    print("No suitable record set with data for EDA.")

## 5. Visualization
Let's visualize the distribution of the numeric field and its normalized values for the selected record set.

> Note: If no data is available, this cell will have no output.

In [ ]:
import matplotlib.pyplot as plt

if record_set_id is not None and numeric_fields:
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    
    df[numeric_field].hist(ax=ax[0])
    ax[0].set_title(f"Distribution of {numeric_field}")
    ax[0].set_xlabel(numeric_field)
    ax[0].set_ylabel("Frequency")
    
    if numeric_field + "_normalized" in filtered_df.columns:
        filtered_df[numeric_field + "_normalized"].hist(ax=ax[1])
        ax[1].set_title(f"Normalized {numeric_field} (filtered)")
        ax[1].set_xlabel("Normalized Value")
        ax[1].set_ylabel("Frequency")
    else:
        ax[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for plotting.")

## 6. Conclusion
In this notebook, we have:
- Loaded and parsed the Croissant metadata for the FAIR² dataset using only the `mlcroissant` library
- Inspected record sets, fields, and IDs, referencing all dataset entities via their `@id`
- Loaded the records into pandas DataFrames
- Performed basic EDA: filter, normalize, and group by
- Plotted distributions of key numeric fields

This workflow can be extended for downstream bioinformatics, statistical analysis, or integration with other data pipelines.

**Remember:** Always reference fields and record sets in code by their `@id` as provided by the Croissant metadata.
